# Model Fitting — Continuation Path

Fit `multidms` models to spike functional-score data along an ascending
`fusionreg` path, warm-starting each step from the previous fit's
parameters. Replicates are fit independently; each replicate has its
own path.

This is a drop-in replacement for `fit_models.ipynb` when the pipeline
config sets `spike.fitting.strategy: "continuation"`. The output
(`fit_collection.pkl`) has the same schema as the independent-fit
notebook, so downstream rules (`evaluate`, `cross_validation`) are
unchanged.

**Outline**
1. Load training functional scores
2. Aggregate per (condition, aa_substitutions) within each replicate
3. Create `multidms.Data` objects (one per replicate)
4. Fit a continuation path across the regularization grid via
   `fit_models_path()`
5. Save the fit collection

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import pickle
import sys

sys.path.insert(0, "notebooks")

import pandas as pd
import multidms
from multidms.model_collection import fit_models_path
from multidms.utils import explode_params_dict

from _common import load_config, build_fit_params

In [ ]:
config_path = "config/config.yaml"
output_dir = None

In [ ]:
config = load_config(config_path)
spike = config["spike"]
fit_config = spike["fitting"]
reference = spike["reference"]

if output_dir is None:
    output_dir = spike.get("output_dir", "results")
os.makedirs(output_dir, exist_ok=True)

## Load training functional scores

In [ ]:
func_score_df = pd.read_csv(
    os.path.join(output_dir, "training_functional_scores.csv")
).fillna({"aa_substitutions": ""})
print(f"Loaded {len(func_score_df):,} variants")

## Create Data objects

Aggregate functional scores per (condition, aa_substitutions) within each
replicate, then create one `multidms.Data` object per replicate.

In [ ]:
data_objects = []
for rep_num, df_rep in func_score_df.groupby("replicate"):
    df_agg = (
        df_rep.groupby(["condition", "aa_substitutions"], dropna=False)
        .agg({"func_score": "mean"})
        .reset_index()
    )
    data = multidms.Data(
        df_agg,
        alphabet=multidms.AAS_WITHSTOP_WITHGAP,
        reference=reference,
        assert_site_integrity=False,
        name=f"rep_{rep_num}",
    )
    data_objects.append(data)
    print(f"rep_{rep_num}: {len(df_agg):,} variants, conditions={data.conditions}")

## Fit models

In [ ]:
fitting_params = build_fit_params(fit_config, data_objects)
print("Fitting parameters:")
for k, v in fitting_params.items():
    if k != "dataset":
        print(f"  {k}: {v}")

In [ ]:
n_steps = len(explode_params_dict(fitting_params))
print(f"Fitting a {n_steps}-step path (sequential by construction)")

n_fit, n_failed, fit_collection_df = fit_models_path(fitting_params)

# Convert dict-valued columns to strings for groupby compatibility
for col in fit_collection_df.columns:
    if fit_collection_df[col].apply(lambda x: isinstance(x, dict)).any():
        fit_collection_df[col] = fit_collection_df[col].apply(str)

print(f"Fit {n_fit} models successfully, {n_failed} failed")

## Save

In [ ]:
output_path = os.path.join(output_dir, "fit_collection.pkl")
with open(output_path, "wb") as f:
    pickle.dump(fit_collection_df, f)
print(f"Saved {output_path} ({len(fit_collection_df)} models)")

In [ ]:
display_cols = ["dataset_name", "fusionreg", "converged", "fit_time"]
display_cols = [c for c in display_cols if c in fit_collection_df.columns]
fit_collection_df[display_cols]